In [ ]:
import polars as pl

In [ ]:
df = pl.read_csv("/home/jens-peter-joost/Coding/cvrp_bench/data/benchmark_vroom.csv")

In [ ]:
df.filter((pl.col("Solver") == "pyvrp") & (pl.col("Solution Quality") >= 1.0)).select("Solution Quality").mean()


In [ ]:
df.filter((pl.col("Solver") == "ortools") & (pl.col("Solution Quality") >= 1.0)).select("Solution Quality").mean()

In [ ]:
df.filter((pl.col("Solver") == "rustvrp") & (pl.col("Solution Quality") >= 1.0)).select("Solution Quality").mean()

In [ ]:
df.filter((pl.col("Solver") == "pyhygese") & (pl.col("Solution Quality") >= 1.0)).select("Solution Quality").mean()

In [ ]:
print(df)

In [ ]:
import matplotlib.pyplot as plt

# Create three plots for time limits 1s, 10s, and 60s
time_limits = [1, 10, 60]
solvers = df.select("Solver").unique().to_series().to_list()

fig, axes = plt.subplots(3, 1, figsize=(15, 15))

for idx, time_limit in enumerate(time_limits):
    ax = axes[idx]
    
    # Filter data for this time limit
    # df_filtered = df.filter(pl.col("Time Limit (s)") == time_limit)
    df_filtered = df
    
    for solver in solvers:
        # Get data for this solver, sorted by size
        solver_data = df_filtered.filter((pl.col("Solver") == solver) & (pl.col("Solution Quality") >= 1)).sort("Size")
        
        sizes = solver_data.select("Size").to_series().to_list()
        qualities = solver_data.select("Solution Quality").to_series().to_list()
        
        ax.plot(sizes, qualities, marker='o', label=solver, markersize=3, alpha=0.7)
    
    ax.set_xlabel("Instance Size")
    ax.set_ylabel("Solution Quality")
    ax.set_title(f"Time Limit: {time_limit}s")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

- Solver, Time Limit, Instances solved, Quality Aware

In [ ]:
summary1 = df.filter(pl.col("Solution Quality") >= 1.0).group_by(["Solver", "Time Limit (s)"
]).agg(pl.col("Instance").count(), pl.col("Solution Quality").mean()).sort(["Time Limit (s)", "Solver"]).rename({"Instance": "#solved"})

In [ ]:
summary1

In [ ]:
summary1[:]

In [ ]:
with pl.Config(tbl_rows=100):
    print(summary1)